In [ ]:
!pip install --upgrade synthcity
!pip show synthcity

In [ ]:
!pip uninstall -y torch torchvision torchaudio torchtext torchmetrics torchao torchtune torchtuples
!pip install torch==2.4.0 torchvision==0.19.0 torchtext==0.18.0

In [ ]:
!pip install torchtuples
!pip uninstall numpy -y
!pip install --upgrade "numpy==2.0.0" "networkx>=3.2" 

In [ ]:
import pandas as pd

df = pd.read_csv(
    '/kaggle/input/dataset-ricoveri/dataset_ricoveri_encode_index.csv',
    sep=';',
    quotechar='"',
    encoding='utf-8',
    on_bad_lines='skip'
)

print(df.head())
print(df.columns)


In [ ]:
# stdlib
import sys
import warnings

warnings.filterwarnings("ignore")

# synthcity absolute
import synthcity.logger as log
from synthcity.plugins import Plugins
from synthcity.plugins.core.dataloader import GenericDataLoader

In [7]:
import pandas as pd

from sklearn.model_selection import train_test_split

cod_regs = df["COD_REG"].dropna().unique()

train_cod, test_cod = train_test_split(
    cod_regs,
    test_size=0.2,
    random_state=3,
    shuffle=True
)

train_df = df[df["COD_REG"].isin(train_cod)].copy()
test_df  = df[df["COD_REG"].isin(test_cod)].copy()

In [ ]:
import numpy as np

codici_scelti = [
    idx
    for idx, group in train_df.groupby("COD_REG")
    if (group["desc_studio_out"] != 2).any()
]

# selezione casuale di 500 pazienti
np.random.seed(0)
riduzione = np.random.choice(codici_scelti, size=3000, replace=False)
print(len(riduzione))

In [ ]:
import numpy as np

codici_scelti = [
    idx
    for idx, group in test_df.groupby("COD_REG")
    if (group["desc_studio_out"] != 2).any()
]

# selezione casuale di 500 pazienti
np.random.seed(40)
riduzione_test = np.random.choice(codici_scelti, size=5000, replace=False)
print(len(riduzione_test))

In [ ]:
df_small_test = test_df[test_df['COD_REG'].isin(riduzione_test)].copy()
loader_test =  GenericDataLoader(df_small_test)
len(loader_test)
loader_test.fillna(0)

In [11]:
df_small = train_df[train_df['COD_REG'].isin(riduzione)].copy()

In [57]:
df_small = df_small.drop(columns=["data_studio_out","desc_studio_out","data_prest","time_step"])

In [12]:
loader = GenericDataLoader(df_small)

In [ ]:
loader.fillna(0)

In [ ]:
syn_model = Plugins().get("ddpm")

In [ ]:
syn_model.fit(loader)

In [16]:
import os

os.makedirs("/kaggle/working/workspace", exist_ok=True)

In [183]:
from synthcity.utils.serialization import save_to_file
from synthcity.plugins import Plugins

save_to_file("/kaggle/working/workspace/ddpm_20000p_ricoveri.pkl",syn_model)

In [ ]:
import os

os.listdir("/kaggle/working")
os.listdir("/kaggle/working/workspace")

In [16]:
synthetic = syn_model.generate(count=17000)

In [ ]:
from synthcity.metrics.eval_statistical import StatisticalEvaluator, ChiSquaredTest

metric = ChiSquaredTest()
print(metric.evaluate(loader_test,synthetic))

In [ ]:
from synthcity.metrics.eval_statistical import JensenShannonDistance

metric = JensenShannonDistance()
print(metric.evaluate(loader_test,synthetic))

In [ ]:
from synthcity.metrics.eval_statistical import InverseKLDivergence

metric = InverseKLDivergence()
print(metric.evaluate(loader_test,synthetic))

In [ ]:
from synthcity.metrics.eval_statistical import PRDCScore

metric = PRDCScore()
print(metric.evaluate(loader_test,synthetic))

In [ ]:
from synthcity.metrics.eval_detection import SyntheticDetectionMLP

metric = SyntheticDetectionMLP()
print(metric.evaluate(loader_test,synthetic))

In [ ]:
from synthcity.metrics.eval_privacy import kAnonymization

metric = kAnonymization()
print(metric.evaluate(loader_test,synthetic))

In [ ]:
from synthcity.metrics.eval_privacy import lDiversityDistinct

metric = lDiversityDistinct()
print(metric.evaluate(loader_test,synthetic))

In [ ]:
from synthcity.metrics.eval_privacy import DeltaPresence

metric = DeltaPresence()
print(metric.evaluate(loader_test,synthetic))

In [ ]:
from synthcity.metrics.eval_privacy import kMap

metric = kMap()
print(metric.evaluate(loader_test,synthetic))

In [ ]:
from synthcity.metrics.eval_privacy import IdentifiabilityScore

metric = IdentifiabilityScore()
print(metric.evaluate(loader_test,synthetic))

In [18]:
import numpy as np
import pandas as pd

def _to_dataframe(loader):
    """
    Converte un GenericDataLoader di synthcity in pandas.DataFrame.
    In synthcity di solito è .dataframe() oppure .to_pandas().
    Proviamo entrambe.
    """
    if hasattr(loader, "dataframe"):
        return loader.dataframe()
    if hasattr(loader, "to_pandas"):
        return loader.to_pandas()
    raise TypeError("Il dataloader non espone dataframe() o to_pandas().")

In [19]:
df_r = _to_dataframe(loader_test).copy()
df_s = _to_dataframe(synthetic).copy()

In [32]:
import os
import matplotlib.pyplot as plt

def plot_feature_distributions(X_real, X_syn, feature_names=None, bins=30, output_dir="/kaggle/working/workspace/feature_plots"):
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Convert DataFrame to numpy if needed
    if hasattr(X_real, "values"):
        X_real = X_real.values
    if hasattr(X_syn, "values"):
        X_syn = X_syn.values

    n_features = X_real.shape[1]
    
    for i in range(n_features):
        plt.figure()
        
        plt.hist(X_real[:, i], bins=bins, alpha=0.5, density=True, label="Real")
        plt.hist(X_syn[:, i], bins=bins, alpha=0.5, density=True, label="Synthetic")
        
        if feature_names:
            title = feature_names[i]
            plt.title(f"Feature: {title}")
            filename = f"{title}.png"
        else:
            title = f"feature_{i}"
            plt.title(f"Feature {i}")
            filename = f"{title}.png"
            
        plt.legend()
        
        # Save the figure
        filepath = os.path.join(output_dir, filename)
        plt.savefig(filepath, bbox_inches="tight", dpi=300)
        
        plt.show()
        plt.close()

In [ ]:
import matplotlib.pyplot as plt

plot_feature_distributions(df_r, df_s, feature_names)

In [37]:
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

scaler = StandardScaler()

X_combined = np.vstack([df_r, df_s])
X_scaled = scaler.fit_transform(X_combined)

X_real_scaled = X_scaled[:len(df_r)]
X_syn_scaled = X_scaled[len(df_s):]

In [ ]:
import seaborn as sns
sns.set_theme(style="ticks", palette="muted")

colors = {
    "Real":      "#4C72B0",  # blu muted seaborn
    "Synthetic": "#DD8452",  # arancione muted seaborn
}

plt.figure()

for lab in ["Real", "Synthetic"]:
    idx = labels == lab
    plt.scatter(X_tsne[idx, 0], X_tsne[idx, 1], label=lab, alpha=0.6)

plt.legend()
plt.title("t-SNE: Real vs Synthetic")
filepath = os.path.join('/kaggle/working/workspace/feature_plots', 'tsne_plot.png')
plt.savefig(filepath, bbox_inches="tight", dpi=300)
plt.show()